# unit01 レッスン: NumPy配列の基本

**このレッスンで作れるようになるもの**: 100万行の数値データを、ループを1つも書かずに「全部2倍する」「条件で絞り込む」「行ごと・列ごとに集計する」処理。

これは機械学習の**すべての土台**です。pandas も scikit-learn も、内部はぜんぶ NumPy 配列で動いています。

- 所要時間: 15〜25分
- 進め方: セルを上から順に実行(`Shift+Enter`)。「書いてみる」セルだけ自分で書く
- 詰まったら: Claude に聞いてOK(答えではなくヒントをくれます)

In [ ]:
import numpy as np

def check(name, actual, expected, hint=""):
    try:
        ok = actual is not None and bool(np.all(np.isclose(np.asarray(actual, dtype=float), np.asarray(expected, dtype=float))))
    except (TypeError, ValueError):
        ok = actual == expected
    if ok:
        print(f"[OK] {name}: 正解!")
    else:
        print(f"[NG] {name}: 期待値 {expected!r} / 実際 {actual!r}")
        if hint:
            print(f"     ヒント: {hint}")
    return ok

print("準備OK! numpy のバージョン:", np.__version__)

---
## 概念1: ベクトル化演算 — 「ループを書かない」という発想転換

### なぜ学ぶか
実務のデータは1万行、100万行が普通です。例えば「全商品の価格を10%値上げする」「全センサー値を摂氏から華氏に変換する」。C# なら `for` ループか LINQ の `Select` で書くところですが、Python の素のループは**遅い**ので、データ処理では NumPy に任せるのが定石です。求人票の「NumPy/pandasでのデータ処理経験」はほぼこれのことです。

### 解説

**NumPy配列(`ndarray`)**は、C# の `T[]` に近い「全要素が同じ型の配列」です。ただし決定的な違いが1つ:

> **演算子が配列全体に一括で効く**(これを*ベクトル化演算*と呼ぶ)

```
C#:      var doubled = arr.Select(x => x * 2).ToArray();  // 要素ごとに関数適用
NumPy:   doubled = arr * 2                                # 配列 × スカラーでおしまい
```

`arr * 2` と書くと、NumPy が内部の C 実装で全要素を一気に処理します。Python でループを書くより**数十〜数百倍速い**。だから「NumPy ではループを書いたら負け」と覚えてください。

まず動くコードを見ます。次のセルを実行して、出力をよく観察してください。

In [ ]:
# GOAL: 配列を作って、ループなしで全要素を演算できることを確認する

# STEP 1: 配列を作る — np.arange(start, stop) は start から stop の「手前」までの連番配列
#         (C# の Enumerable.Range(start, count) と違い、第2引数は「個数」ではなく「終了値(含まない)」)
prices = np.arange(100, 600, 100)   # 100から600の手前まで100刻み
print("元の配列 :", prices)
print("型(dtype):", prices.dtype)   # 全要素が同じ型 — C# の int[] と同じ感覚

# STEP 2: 全要素への一括演算 — ループ不要
raised = prices * 1.1               # 全価格を10%値上げ
print("10%値上げ:", raised)

# STEP 3: 集計 — 合計・平均・最大にも専用メソッドがある(LINQ の Sum()/Average()/Max() 相当)
print("合計:", prices.sum(), " 平均:", prices.mean(), " 最大:", prices.max())

### 予測してみよう

次のセルは `np.arange(2, 11, 2)`(2から10までの偶数)に対して `** 2`(2乗)を計算します。

**実行する前に**、出力がどうなるか頭の中で予測してください。特に「配列の長さは何個になるか」「最後の要素は何か」。

In [ ]:
# 予測してから実行!
evens = np.arange(2, 11, 2)
print("元の配列:", evens)
print("2乗     :", evens ** 2)

予測は当たりましたか? `11` は含まれない(終了値は含まない)ので配列は5個、最後は `10 ** 2 = 100` です。ここを間違えると「1個足りない」バグになるので、C# の `Range(start, count)` との違いは体に入れておきましょう。

### 書いてみる

**課題**: `1 から 8 までの連番配列`を作り、**全要素を100倍**した配列を `result1` に入れてください(期待値: `[100, 200, ..., 800]`)。

ヒント(概念レベル): 「連番を作る関数」と「一括演算」を組み合わせるだけ。ループは書かない。

In [ ]:
result1 = None
# ここに書く(result1 に代入する)


check("概念1: ベクトル化演算", result1,
      np.array([100, 200, 300, 400, 500, 600, 700, 800]),
      hint="np.arange の終了値は「含まれない」ことに注意。8まで欲しいなら stop はいくつ?")

---
## 概念2: ブールマスク — 条件で絞り込む

### なぜ学ぶか
「売上が10万円以上の店舗だけ」「外れ値(異常に大きい値)を除外」「テストデータのうち正解したものだけ」— データ分析の仕事は**絞り込み**の連続です。C# で言えば LINQ の `Where` ですが、NumPy には独特の書き方があり、これが読めないと他人のデータ処理コードが一切読めません。

### 解説

NumPy で `arr > 50` のように**配列と値を比較すると、bool の配列**が返ってきます:

```python
arr = np.array([30, 60, 90])
arr > 50    # → array([False, True, True])
```

この bool 配列を**マスク(仮面)**と呼びます。そしてマスクを添字 `arr[マスク]` に入れると、**True の位置の要素だけ**が取り出せます。

```
C#:      arr.Where(x => x > 50).ToArray()
NumPy:   arr[arr > 50]
```

「条件式を作る」→「それを添字に入れる」の2段構えです。次のセルで中間状態(マスクそのもの)を見てみましょう。

In [ ]:
# GOAL: マスクが「bool配列」であることを目で確認し、絞り込みに使う

scores = np.array([45, 82, 67, 91, 38, 74])

# STEP 1: 条件式はbool配列を返す(これがマスク)
mask = scores >= 70
print("マスク       :", mask)

# STEP 2: マスクを添字に入れると True の位置だけ取り出せる(LINQ の Where 相当)
print("70点以上     :", scores[mask])

# STEP 3: マスクの合計 = True の個数(True=1, False=0 として数えられる — Count() 相当)
print("70点以上の人数:", mask.sum())

### 予測してみよう

次のセルでは `scores % 2 == 0`(偶数かどうか)のマスクを使います。実行前に予測: **偶数の要素はどれで、何個ありますか?**(scores = [45, 82, 67, 91, 38, 74])

In [ ]:
# 予測してから実行!
print("偶数だけ:", scores[scores % 2 == 0])
print("偶数の個数:", (scores % 2 == 0).sum())

### 書いてみる

**課題**: `temperatures`(1週間の気温)から **25度を超える日の気温だけ**を取り出して `result2` に入れてください(期待値: `[28.1, 26.4, 30.2]`)。

ヒント(概念レベル): 「条件式でマスクを作る」→「添字に入れる」の2段構え。1行で書けます。

In [ ]:
temperatures = np.array([22.5, 28.1, 24.9, 26.4, 21.0, 30.2, 25.0])

result2 = None
# ここに書く(result2 に代入する)


check("概念2: ブールマスク", result2, np.array([28.1, 26.4, 30.2]),
      hint="「超える」なので >= ではなく > 。25.0ちょうどの日は含まれない")

---
## 概念3: ブロードキャストと axis — 2次元データを一撃で処理する

### なぜ学ぶか
実データは「行=サンプル、列=項目」の**表**の形をしています(生徒×科目の得点表、店舗×月の売上表)。機械学習の前処理では「列ごとに平均を引く」「行ごとに合計で割る」操作が頻出します(特徴量のスケーリング — unit03以降で毎回使います)。C# なら二重ループですが、NumPy は**形状の違う配列同士の演算を自動で引き伸ばして**くれます。これが**ブロードキャスト**です。

### 解説

2次元配列には `shape`(形)があります。`shape (3, 4)` = 3行4列。

**ブロードキャスト**: `(3, 4)` の行列に `(4,)` の1次元配列を足すと、NumPy は1次元配列を「3行分コピーしたつもり」で各行に足してくれます。ループ不要。

**axis(軸)**: 集計の方向指定です。ここが最初の関門:
- `axis=0` → **行方向につぶす** = **列ごと**の集計(各列の平均など)
- `axis=1` → **列方向につぶす** = **行ごと**の集計(各行の合計など)

「axis=でつぶす方向を指定する」と覚えます。C# で言えば「二重ループのどちらを外側にするか」に相当しますが、1引数で済みます。

In [ ]:
# GOAL: ブロードキャストと axis の動きを目で確認する

# 生徒3人 × 科目4つ の得点表
table = np.array([[80, 60, 70, 90],
                  [50, 95, 65, 70],
                  [75, 80, 85, 60]])
print("shape:", table.shape)   # (3, 4) = 3行4列

# STEP 1: ブロードキャスト — (4,) のボーナス点を「各行に」足す
bonus = np.array([5, 0, 10, 0])
print("ボーナス加算後:\n", table + bonus)

# STEP 2: axis=0 → 行方向につぶす = 「科目ごと(列ごと)」の平均
print("科目ごとの平均:", table.mean(axis=0))   # 結果は shape (4,)

# STEP 3: axis=1 → 列方向につぶす = 「生徒ごと(行ごと)」の平均
print("生徒ごとの平均:", table.mean(axis=1))   # 結果は shape (3,)

### 予測してみよう

`table.sum(axis=0)` と `table.sum(axis=1)`、**結果の要素数はそれぞれ何個**でしょう? 「つぶす方向」から考えてから実行してください。

In [ ]:
# 予測してから実行!
print("axis=0 の合計:", table.sum(axis=0), " ← 要素数は?")
print("axis=1 の合計:", table.sum(axis=1), " ← 要素数は?")

axis=0(行をつぶす)→ 列の数だけ残るので4個。axis=1(列をつぶす)→ 行の数だけ残るので3個。「**つぶした軸が消える**」と覚えると迷いません。

### 書いてみる

**課題**: `table` の**科目ごとの平均点**(axis はどっち?)を計算し、**各得点からその科目の平均を引いた表**を `result3` に入れてください(=科目の難易度差を取り除く「中心化」。unit03 の前処理で実際に使います)。

ヒント(概念レベル): 集計してからブロードキャストで引き算。2段階で書いてOK。

In [ ]:
result3 = None
# ここに書く(result3 に代入する)


check("概念3: ブロードキャスト", result3,
      np.array([[ 11.66666667, -18.33333333,  -3.33333333,  16.66666667],
                [-18.33333333,  16.66666667,  -8.33333333,  -3.33333333],
                [  6.66666667,   1.66666667,  11.66666667, -13.33333333]]),
      hint="「科目ごと」はどちらの axis? 集計結果をそのまま table から引けば各行にブロードキャストされる")

---
## 振り返り(1〜2文でOK — このセルを編集して書き込んでください)

- **今日学んだことを自分の言葉で**:
- **難しかったこと(あれば)**:

(この記述はセッション終了時にチューターが学習ノートとスキルレベル判定に使います)

## まとめと次へ

| 概念 | 一言で | C#で言うと |
|------|--------|-----------|
| ベクトル化演算 | `arr * 2` で全要素一括 | `Select` がループなしで効く |
| ブールマスク | `arr[arr > 50]` で絞り込み | `Where` |
| ブロードキャスト / axis | 形の違う配列を自動で引き伸ばし、`axis=` でつぶす方向指定 | 二重ループが1引数に |

**この先どこで使うか**: unit02 の pandas は内部が NumPy 配列そのもの(マスクでのフィルタが再登場)。unit03〜05 の特徴量スケーリングは今日の「中心化」の実戦版です。

**次**: 演習 `ex01_arrays.py` へ。lesson を見ながらで OK。テストは
`python -m pytest courses/ml-intro/unit01-numpy-basics/tests/test_ex01.py -q`